# Slack Exporter

Exports all private Slack channels your bot token is a member of in the [official Slack export format](https://slack.com/help/articles/201658943-Export-your-workspace-data), then packages everything into a ZIP archive you can download.

## Required OAuth scopes

| Scope | Purpose |
|-------|---------|
| `groups:read` | List private channels the bot is a member of |
| `groups:history` | Read private channel message history |
| `users:read` | Fetch workspace user list |
| `emoji:read` | Fetch custom workspace emoji |
| `files:read` | Download file attachments |
| `metadata.message:read` | Include structured message metadata |

## Setup
1. Create a [Slack app](https://api.slack.com/apps) and add the OAuth scopes above.
2. Install the app to your workspace.
3. Invite the bot to each private channel you want to export: `/invite @your-bot`
4. Add your bot token to [Colab Secrets](https://medium.com/@d.aniruddha/googles-colab-secret-feature-e1ef19c4e7c5) as `SLACK_TOKEN` (key icon 🔑 in the left sidebar), or paste it when prompted below.

In [ ]:
!pip install -q slack-sdk tqdm requests

import json
import os
import time
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
from tqdm.auto import tqdm
import requests

print('Setup complete.')

## 1. Enter your Slack token

**Recommended:** Add `SLACK_TOKEN` to [Colab Secrets](https://medium.com/@d.aniruddha/googles-colab-secret-feature-e1ef19c4e7c5) (the 🔑 icon in the left sidebar). The cell below loads it automatically.

Otherwise you will be prompted to paste the token in.

In [ ]:
import getpass

try:
    from google.colab import userdata
    SLACK_TOKEN = userdata.get('SLACK_TOKEN')
    if SLACK_TOKEN:
        print('Token loaded from Colab Secrets.')
    else:
        raise ValueError('Secret not set')
except Exception:
    SLACK_TOKEN = (
        os.environ.get('SLACK_TOKEN')
        or getpass.getpass('Paste your Slack token (xoxb-...): ')
    )

assert SLACK_TOKEN and SLACK_TOKEN.startswith(('xoxb-', 'xoxp-')), \
    'Token should start with xoxb- or xoxp-'
print('Token OK.')

## 2. Load the exporter

Run this cell once to define the `SlackExporter` class.

In [ ]:
class SlackExporter:
    """Exports a Slack workspace to the official export format."""

    def __init__(self, token: str, output_dir: str = None, download_files: bool = True):
        self.client = WebClient(token=token)
        self.out = Path(output_dir) if output_dir else None
        self.download_files = download_files
        # Reuse one session for all file downloads; auth header sent automatically.
        self._http = requests.Session()
        self._http.headers["Authorization"] = f"Bearer {token}"

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _call(self, method: str, **kwargs):
        """Wrapper around the Slack SDK that handles rate-limit retries."""
        while True:
            try:
                fn = getattr(self.client, method)
                return fn(**kwargs)
            except SlackApiError as exc:
                if exc.response["error"] == "ratelimited":
                    wait = int(exc.response.headers.get("Retry-After", 60))
                    ts = datetime.now().strftime("%I:%M:%S %p")
                    tqdm.write(f"    [{ts}] [rate limit] waiting {wait}s …")
                    time.sleep(wait)
                else:
                    raise

    def _paginate(self, method: str, result_key: str, **kwargs):
        """Yield all items from a paginated Slack API endpoint."""
        cursor = None
        while True:
            if cursor:
                kwargs["cursor"] = cursor
            result = self._call(method, **kwargs)
            yield from result.get(result_key, [])
            cursor = result.get("response_metadata", {}).get("next_cursor")
            if not cursor:
                break

    # ------------------------------------------------------------------
    # Data fetchers
    # ------------------------------------------------------------------

    def fetch_users(self):
        users = []
        with tqdm(desc="Fetching users", unit=" user", leave=True) as pbar:
            for user in self._paginate("users_list", "members", limit=200):
                users.append(user)
                pbar.update(1)
        return users

    def fetch_channels(self, name: str = None):
        channels = []
        with tqdm(desc="Fetching channels", unit=" ch", leave=True) as pbar:
            for ch in self._paginate(
                "conversations_list",
                "channels",
                types="private_channel",
                exclude_archived=False,
                limit=200,
            ):
                if name is None or ch["name"] == name:
                    channels.append(ch)
                    pbar.update(1)
        return channels

    def fetch_members(self, channel_id: str):
        try:
            return list(self._paginate(
                "conversations_members", "members",
                channel=channel_id, limit=200,
            ))
        except SlackApiError as exc:
            tqdm.write(f"    [warn] could not fetch members: {exc.response['error']}")
            return []

    def fetch_history(self, channel_id: str):
        """Return all messages in a channel (oldest first)."""
        messages = []
        try:
            with tqdm(desc="  messages", unit=" msg", leave=False) as pbar:
                for msg in self._paginate(
                    "conversations_history", "messages",
                    channel=channel_id, limit=200, include_all_metadata=True,
                ):
                    messages.append(msg)
                    pbar.update(1)
        except SlackApiError as exc:
            tqdm.write(f"    [warn] could not read history: {exc.response['error']}")
        # conversations_history returns newest-first; reverse to oldest-first
        messages.reverse()
        return messages

    def fetch_replies(self, channel_id: str, thread_ts: str):
        """
        Return reply messages for a thread (excludes the parent message
        so it is not duplicated in the output).
        """
        try:
            gen = self._paginate(
                "conversations_replies", "messages",
                channel=channel_id, ts=thread_ts, limit=200,
            )
            next(gen, None)  # skip parent message (always first item on page 1)
            return list(gen)
        except SlackApiError as exc:
            tqdm.write(f"    [warn] thread {thread_ts}: {exc.response['error']}")
            return []

    # ------------------------------------------------------------------
    # File download helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _collect_files(messages: list) -> list:
        """Return a deduplicated list of file objects found in messages."""
        seen, files = set(), []
        for msg in messages:
            for f in msg.get("files", []):
                fid = f.get("id")
                # Skip deleted files (Slack marks them as "tombstone")
                if fid and fid not in seen and f.get("mode") != "tombstone":
                    seen.add(fid)
                    files.append(f)
        return files

    def _download_file(self, url: str, dest: Path) -> bool:
        """Stream a private Slack file to *dest*. Returns True on success."""
        if dest.exists():
            return True  # already downloaded (resumable re-runs)
        try:
            resp = self._http.get(url, stream=True, timeout=60)
            resp.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in resp.iter_content(chunk_size=65536):
                    fh.write(chunk)
            return True
        except Exception as exc:
            tqdm.write(f"    [warn] download failed {dest.name}: {exc}")
            dest.unlink(missing_ok=True)  # remove any partial write
            return False

    def download_channel_files(self, messages: list, ch_dir: Path):
        """Download all file attachments referenced in *messages* to *ch_dir*/_files/."""
        files = self._collect_files(messages)
        if not files:
            return
        files_dir = ch_dir / "_files"
        files_dir.mkdir(exist_ok=True)
        bar = tqdm(files, desc="  files", unit=" file", leave=False)
        for f in bar:
            url = f.get("url_private_download") or f.get("url_private")
            if not url:
                continue
            name = f.get("name") or f.get("id", "unknown")
            dest = files_dir / f"{f['id']}-{name}"
            bar.set_postfix_str(name[:40])
            self._download_file(url, dest)
            # Relative to the channel dir so the path works regardless of
            # where the export folder lives.
            f["local_path"] = f"_files/{dest.name}"

    # ------------------------------------------------------------------
    # Emoji helpers
    # ------------------------------------------------------------------

    def fetch_emoji(self) -> dict:
        """Return the workspace custom emoji map {name: url_or_alias}."""
        result = self._call("emoji_list")
        return result.get("emoji", {})

    def download_emoji(self, emoji: dict):
        """Download all non-alias custom emoji images to __emoji/ in the output dir."""
        # Aliases point to other emoji names ("alias:other-name"), not URLs.
        to_download = {name: url for name, url in emoji.items()
                       if not url.startswith("alias:")}
        if not to_download:
            return
        emoji_dir = self.out / "__emoji"
        emoji_dir.mkdir(exist_ok=True)
        bar = tqdm(to_download.items(), desc="Emoji", unit=" emoji", leave=True)
        for name, url in bar:
            ext = Path(url.split("?")[0]).suffix or ".png"
            dest = emoji_dir / f"{name}{ext}"
            bar.set_postfix_str(name[:40])
            self._download_file(url, dest)

    # ------------------------------------------------------------------
    # Formatting helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _format_channel(channel: dict, members: list) -> dict:
        """Trim a channel object to the fields present in official exports."""
        blank_set = {"value": "", "creator": "", "last_set": 0}
        return {
            "id": channel.get("id"),
            "name": channel.get("name"),
            "created": channel.get("created"),
            "creator": channel.get("creator"),
            "is_archived": channel.get("is_archived", False),
            "is_general": channel.get("is_general", False),
            "members": members,
            "topic": channel.get("topic", blank_set),
            "purpose": channel.get("purpose", blank_set),
        }

    @staticmethod
    def _ts_to_day(ts: str) -> str:
        """Convert a Slack timestamp ('1234567890.123456') to 'YYYY-MM-DD'."""
        return datetime.fromtimestamp(float(ts), tz=timezone.utc).strftime("%Y-%m-%d")

    @staticmethod
    def _group_by_day(messages: list) -> dict:
        """Group messages into {YYYY-MM-DD: [msg, ...]} sorted by timestamp."""
        by_day: dict = defaultdict(list)
        for msg in messages:
            day = SlackExporter._ts_to_day(msg.get("ts", "0"))
            by_day[day].append(msg)
        for day in by_day:
            by_day[day].sort(key=lambda m: float(m.get("ts", 0)))
        return dict(sorted(by_day.items()))

    # ------------------------------------------------------------------
    # Write helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _write_json(path: Path, data):
        with open(path, "w", encoding="utf-8") as fh:
            json.dump(data, fh, indent=4, ensure_ascii=False)

    # ------------------------------------------------------------------
    # Main export
    # ------------------------------------------------------------------

    def export(self, channel: str = None) -> Path:
        self.out.mkdir(parents=True, exist_ok=True)

        # 1. Users
        users = self.fetch_users()
        self._write_json(self.out / "users.json", users)

        # 2. Custom emoji
        emoji = self.fetch_emoji()
        self._write_json(self.out / "emoji.json", emoji)
        if self.download_files:
            self.download_emoji(emoji)

        # 3. Channels + messages
        channels = self.fetch_channels(name=channel)
        if channel and not channels:
            raise ValueError(f"Channel '{channel}' not found or not accessible.")
        channels_meta = []

        ch_bar = tqdm(channels, desc="Channels", unit=" ch")
        for ch in ch_bar:
            cid = ch["id"]
            name = ch["name"]
            ch_bar.set_postfix_str(f"#{name}")

            members = self.fetch_members(cid)
            channels_meta.append(self._format_channel(ch, members))

            # --- messages + thread replies ---
            messages = self.fetch_history(cid)

            seen_ts: set = {m["ts"] for m in messages}
            thread_parents = [
                m for m in messages
                if m.get("reply_count", 0) > 0
                and m.get("thread_ts") == m.get("ts")
            ]
            if thread_parents:
                for parent in tqdm(
                    thread_parents,
                    desc="  threads",
                    unit=" thread",
                    leave=False,
                ):
                    for reply in self.fetch_replies(cid, parent["thread_ts"]):
                        if reply["ts"] not in seen_ts:
                            messages.append(reply)
                            seen_ts.add(reply["ts"])

            # Sort all messages (oldest first) before grouping
            messages.sort(key=lambda m: float(m.get("ts", 0)))

            # --- download attachments (patches local_path into message dicts) ---
            ch_dir = self.out / name
            ch_dir.mkdir(exist_ok=True)
            if self.download_files:
                self.download_channel_files(messages, ch_dir)

            # --- write daily files (local_path fields already set above) ---
            by_day = self._group_by_day(messages)
            for day, day_msgs in by_day.items():
                self._write_json(ch_dir / f"{day}.json", day_msgs)

        self._write_json(self.out / "channels.json", channels_meta)

        # 4. ZIP archive
        zip_path = self.out.with_suffix(".zip")
        all_files = [fp for fp in sorted(self.out.rglob("*")) if fp.is_file()]
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in tqdm(all_files, desc="Zipping", unit=" file"):
                zf.write(fp, fp.relative_to(self.out.parent))

        print(f"\nDone.")
        print(f"  Directory : {self.out}")
        print(f"  Archive   : {zip_path}")
        return zip_path

print('SlackExporter ready.')

## 3. Mount Google Drive

The export will be saved directly to your Google Drive. You'll be prompted to authorize access when you run this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive where exports will be saved
DRIVE_FOLDER = '/content/drive/MyDrive/slack_exports'
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f'Drive mounted. Exports will be saved to: {DRIVE_FOLDER}')

## 4. Configure the export

Fill in the form below — Colab renders `#@param` annotations as interactive inputs.

In [ ]:
# @title Export settings
# @markdown Leave **Channel name** blank to export all accessible channels.

channel_name = ""  # @param {type:"string", placeholder:"e.g. engineering"}
download_files = True  # @param {type:"boolean"}

# Normalise: empty string → None (export everything)
CHANNEL = channel_name.strip() or None
DOWNLOAD_FILES = download_files

# Output goes directly into your Drive folder (a .zip is also created there)
OUTPUT_DIR = f"{DRIVE_FOLDER}/slack_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Channel  : {CHANNEL or '(all channels)'}")
print(f"Files    : {DOWNLOAD_FILES}")
print(f"Output   : {OUTPUT_DIR}")

## (Optional) Preview accessible channels

Run this cell to see which channels your token can access before running the full export.

In [ ]:
_preview = SlackExporter(token=SLACK_TOKEN)
_channels = _preview.fetch_channels()

print(f"\n{'ID':<12} {'Archived':<9} Name")
print("-" * 45)
for _ch in sorted(_channels, key=lambda c: c['name']):
    _archived = 'yes' if _ch.get('is_archived') else ''
    print(f"{_ch['id']:<12} {_archived:<9} #{_ch['name']}")
print(f"\n{len(_channels)} channel(s)")

## Run the export

This may take a while for large workspaces. Progress bars show below each cell while it runs.

In [ ]:
exporter = SlackExporter(
    token=SLACK_TOKEN,
    output_dir=OUTPUT_DIR,
    download_files=DOWNLOAD_FILES,
)
zip_path = exporter.export(channel=CHANNEL)

In [ ]:
print(f"Export saved to Google Drive:")
print(f"  Directory : {exporter.out}")
print(f"  ZIP       : {zip_path}")
print()
print("Find it in your Drive under: My Drive > slack_exports/")